# 00_smoke - QLoRA smoke test (Colab free T4)

**Purpose:** prove the full path in <15 min: install -> 4-bit load -> LoRA -> 10 dummy samples -> 20 train steps -> adapter save/reload hash match.

**Run order:** 00_smoke -> 01_train -> 02_merge_export (see `RUNBOOK.md`).

Every cell fails loudly. A red `SMOKE FAIL:` message means STOP and read it.

In [ ]:
# C1 - pinned installs + environment check (fails loudly)
import sys, subprocess

PINS = [
    "transformers==4.53.2", "peft==0.17.1", "bitsandbytes==0.46.1",
    "accelerate==1.9.0", "datasets==4.0.0", "sentencepiece", "jinja2",
]
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PINS])
except Exception as e:
    raise SystemExit(f"INSTALL FAIL: pip could not install pinned versions: {e}")

try:
    import torch, transformers, peft, bitsandbytes, accelerate, datasets, jinja2
except Exception as e:
    raise SystemExit(f"INSTALL FAIL: import error after install: {e}")

print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("accelerate  :", accelerate.__version__)
print("datasets    :", datasets.__version__)
print("jinja2      :", jinja2.__version__)

if not torch.cuda.is_available():
    raise SystemExit("CUDA FAIL: no GPU visible. Colab menu: Runtime > Change runtime type > T4 GPU, then rerun from C1.")
print("GPU         :", torch.cuda.get_device_name(0))
print("GPU memory  :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

# NOTE: if transformers==4.53.2 has no native LFM2 config, from_pretrained falls
# back to trust_remote_code=True (set on every load below). Logged in docs/STATUS.md.

In [ ]:
# C2 - download base model (~2.4 GB). Weights land in Colab /content only; never committed.
from huggingface_hub import snapshot_download
import os

MODEL_ID = "LiquidAI/LFM2.5-1.2B-Thinking"
BASE_DIR = "/content/base_model"

try:
    snap = snapshot_download(
        MODEL_ID,
        local_dir=BASE_DIR,
        allow_patterns=[
            "config.json", "model.safetensors.index.json", "model*.safetensors",
            "tokenizer*", "*template*",
        ],
    )
except Exception as e:
    raise SystemExit(f"SMOKE FAIL: snapshot_download({MODEL_ID!r}) failed: {e}")

total = 0
for dp, _, fs in os.walk(snap):
    for f in fs:
        total += os.path.getsize(os.path.join(dp, f))
print(f"downloaded: {snap} ({total/1e9:.2f} GB)")
if total < 1e9:
    raise SystemExit(f"SMOKE FAIL: download suspiciously small ({total} bytes) - weights missing")

In [ ]:
from pathlib import Path
import hashlib

def find_repo_file(rel):
    # Self-contained: works whether cwd is the colab/ folder itself, the repo root,
    # /content after a plain colab/ upload, or /content/Ira after a clone.
    rel = rel[len("colab/"):] if rel.startswith("colab/") else rel
    aliases = {
        "eval/prompts.jsonl": "data/eval_prompts.jsonl",
        "data/sft/ban_strings.txt": "data/ban_strings.txt",
        "data/sft/identity.jsonl": "data/identity.jsonl",
    }
    variants = [rel, aliases[rel]] if rel in aliases else [rel]
    bases = [Path.cwd(), *list(Path.cwd().parents)[:4],
             Path("/content/colab"), Path("/content/Ira"), Path("/content/Ira/colab"), Path("/content")]
    for b in bases:
        for v in variants:
            for cand in (b / v, b / "colab" / v):
                if cand.is_file():
                    return cand
    raise SystemExit(
        f"PATH FAIL: {rel} not found (tried {variants} from {Path.cwd()} and /content). "
        "Upload the WHOLE colab/ folder — configs/, data/ and the notebooks must stay together.")
TEMPLATE_PATH = find_repo_file("colab/configs/chat_template.txt")
TEMPLATE_TEXT = TEMPLATE_PATH.read_text()
if not TEMPLATE_TEXT.strip():
    raise SystemExit(f"TEMPLATE FAIL: {TEMPLATE_PATH} is empty")
if "{{ .Prompt }}" in TEMPLATE_TEXT:
    raise SystemExit(f"TEMPLATE FAIL: {TEMPLATE_PATH} contains the Ollama serving template, wrong source")
assert "<|im_start|>" in TEMPLATE_TEXT and "<think>" in TEMPLATE_TEXT, \
    f"TEMPLATE FAIL: {TEMPLATE_PATH} missing role/think markers - wrong source"


from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_DIR,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_DIR, trust_remote_code=True)
tokenizer.chat_template = TEMPLATE_TEXT
print("base loaded:", BASE_DIR, "| params:", f"{model.num_parameters():,}")

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules="all-linear", task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
import json
from datasets import Dataset

TO, TC = "<" + "think>", "</" + "think>"

dummy_messages = [
    [
        {"role": "system", "content": "You are Ira. Talk like yourself."},
        {"role": "user", "content": f"dummy user message {i}"},
        {"role": "assistant",
         "content": f"{TO}ok quick dummy thought {i}{TC}dummy reply {i}"},
    ]
    for i in range(10)
]

rendered = []
for msgs in dummy_messages:
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    ids = tokenizer(text, add_special_tokens=False).input_ids
    assert len(ids) > 20, f"SMOKE FAIL: rendered sample too short: {text!r}"
    rendered.append({"text": text})

smoke_ds = Dataset.from_list(rendered)
print(smoke_ds)
print("--- sample 0, verbatim ---")
print(smoke_ds[0]["text"])

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

OUT_DIR = "/content/smoke_out"
args = TrainingArguments(
    output_dir=OUT_DIR,
    max_steps=20,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    logging_steps=5,
    report_to="none",
    save_strategy="no",
    remove_unused_columns=False,
)

def tokenize(batch):
    out = tokenizer(batch["text"], truncation=True, max_length=512,
                    padding="max_length", add_special_tokens=False)
    out["labels"] = [list(x) for x in out["input_ids"]]
    return out

smoke_tok = smoke_ds.map(tokenize, batched=True, remove_columns=["text"])
trainer = Trainer(
    model=model, args=args, train_dataset=smoke_tok,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
try:
    result = trainer.train()
except Exception as e:
    raise SystemExit(f"SMOKE FAIL: trainer.train() raised {type(e).__name__}: {e}")
print("final loss:", result.training_loss)

In [ ]:
import hashlib, shutil, os, torch

log = trainer.state.log_history
loss_by_step = {e["step"]: e["loss"] for e in log if "loss" in e}
try:
    pre = [s for s in loss_by_step if s <= 5]
    assert pre, f"SMOKE FAIL: no loss logged at/before step 5: {loss_by_step}"
    assert 20 in loss_by_step, f"SMOKE FAIL: no loss logged at step 20: {loss_by_step}"
    l5 = min(loss_by_step[s] for s in pre)
    l20 = loss_by_step[20]
    assert l20 < l5, f"SMOKE FAIL: loss did not decrease (step5={l5:.4f} >= step20={l20:.4f})"
    print(f"loss step5={l5:.4f} -> step20={l20:.4f}  OK")

    ADAPTER_DIR = "/content/smoke_adapter"
    shutil.rmtree(ADAPTER_DIR, ignore_errors=True)
    trainer.model.save_pretrained(ADAPTER_DIR)
    assert os.path.isfile(f"{ADAPTER_DIR}/adapter_model.safetensors"), "SMOKE FAIL: adapter_model.safetensors missing after save"
    assert os.path.isfile(f"{ADAPTER_DIR}/adapter_config.json"), "SMOKE FAIL: adapter_config.json missing after save"

    def adapter_hash(path):
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                h.update(chunk)
        return h.hexdigest()

    h1 = adapter_hash(f"{ADAPTER_DIR}/adapter_model.safetensors")

    from peft import PeftModel
    fresh = AutoModelForCausalLM.from_pretrained(
        BASE_DIR, quantization_config=bnb_config, device_map="auto",
        trust_remote_code=True, torch_dtype=torch.float16,
    )
    reloaded = PeftModel.from_pretrained(fresh, ADAPTER_DIR)
    reloaded.save_pretrained("/content/smoke_adapter_reloaded")
    h2 = adapter_hash("/content/smoke_adapter_reloaded/adapter_model.safetensors")
    assert h1 == h2, f"SMOKE FAIL: adapter hash mismatch after reload\n  saved:    {h1}\n  reloaded: {h2}"
    print(f"adapter sha256 match: {h1[:16]}...")
    del fresh, reloaded
    torch.cuda.empty_cache()
    print()
    print("SMOKE PASS - safe to run 01_train.ipynb")
except SystemExit:
    raise
except Exception as e:
    raise SystemExit(f"SMOKE FAIL: {type(e).__name__}: {e}")